# ExecuTorch-Spyre — Part 1: Generate the `.pte`

Starts from a PyTorch model, runs the ExecuTorch-Spyre ahead-of-time (AOT) preprocessing
pipeline, and writes a `.pte` artifact ready for on-device execution.

The AOT compile step initializes compiler/driver resources. Run cell 6 (`free_runtime()`)
at the end of this notebook and shut down the kernel before opening **Part 2** (`02_run_on_device.ipynb`).

## 1. Verify environment and package imports

Validates that the required packages are present in the active Python environment.
Ensure you are running under a dedicated Python 3.12 kernel with `torch==2.13.0`,
`executorch==1.5.1`, and `executorch-spyre` installed (`pip install -e .`).

In [ ]:
import importlib.util, json, os, sys
from pathlib import Path

# Locate REPO_ROOT: try (in order):
#   1. EXECUTORCH_SPYRE_REPO_ROOT env var
#   2. Notebook file location (VS Code / JupyterLab expose __vsc_ipynb_file__)
#   3. Upward traversal from CWD

def _find_repo_root() -> Path:
    candidates = []
    # Notebook's own directory (VS Code sets __vsc_ipynb_file__; JupyterLab sets __session__)
    nb_file = globals().get("__vsc_ipynb_file__") or globals().get("__session__")
    if nb_file:
        candidates.append(Path(nb_file).resolve().parent)
    # CWD fallback
    candidates.append(Path.cwd().resolve())
    for start in candidates:
        for directory in [start, *start.parents]:
            if (directory / "pyproject.toml").exists():
                return directory
    return candidates[0]  # will fail the assert below with a clear message

REPO_ROOT = Path(os.environ.get("EXECUTORCH_SPYRE_REPO_ROOT", "") or _find_repo_root()).resolve()
assert (REPO_ROOT / "pyproject.toml").exists(), (
    f"Repo not found at {REPO_ROOT}.\n"
    "Set the EXECUTORCH_SPYRE_REPO_ROOT environment variable to the executorch-spyre repo root, "
    "or open this notebook from inside the repo directory."
)

manifest_path = REPO_ROOT / "compatibility-manifest.json"
pins = json.loads(manifest_path.read_text()) if manifest_path.exists() else {"torch": "2.13.0", "executorch": "1.5.1"}

import torch
assert torch.__version__.split("+")[0] == pins["torch"], (
    f"Wrong torch version: need {pins['torch']}, got {torch.__version__}. "
    "Please ensure you are using the configured virtual environment kernel."
)

import executorch
import torch_spyre  # Registers the Spyre device and runtime
from executorch.backends.spyre import SpyrePartitioner

print("Repository: ", REPO_ROOT)
print("Python:     ", sys.executable)
print("torch:      ", torch.__version__)
print("executorch: ", getattr(executorch, "__version__", "installed"))
print("torch_spyre:", getattr(torch_spyre, "__version__", "installed"))
print("Environment verification OK")

## 2. Set up workspace and compiler settings

Sets up the workspace scratch directory and sets `FLEX_DEVICE=MOCK` so the Spyre compiler
can run AOT preprocessing without requiring exclusive physical VFIO hardware access.

In [ ]:
# Configurable scratch directory (default: REPO_ROOT / "work")
work_home = Path(os.environ.get("EXECUTORCH_SPYRE_WORK_HOME", REPO_ROOT / "work"))
work_home.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HOME", str(work_home))
os.environ.setdefault("XDG_CACHE_HOME", str(work_home / ".cache"))
os.environ["FLEX_DEVICE"] = os.environ.get("FLEX_DEVICE", "MOCK")

print("Workspace home:", work_home)
print("FLEX_DEVICE:   ", os.environ["FLEX_DEVICE"])

## 3. Define the model, inputs, and CPU reference

A fully delegated 64×64 fp16 matmul. The CPU reference (fp32 accumulation, cast
to fp16) is written to `ref.bin` for the on-device comparison in Part 2.

In [ ]:
class MatMul(torch.nn.Module):
    def forward(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return torch.matmul(a, b)

M = K = N = 64
generator = torch.Generator().manual_seed(0)
a = torch.randn(M, K, dtype=torch.float16, generator=generator)
b = torch.randn(K, N, dtype=torch.float16, generator=generator)
reference = (a.float() @ b.float()).to(torch.float16)
model = MatMul().eval()

print("a:", tuple(a.shape), a.dtype)
print("b:", tuple(b.shape), b.dtype)
print("reference[0, :4]:", reference[0, :4].tolist())

## 4. Preprocess the graph and generate the `.pte`

`SpyrePartitioner` selects the graph for Spyre delegation. `SpyreBackend.preprocess()`
invokes the Spyre compiler (via `FLEX_DEVICE=MOCK`) and embeds the resulting
SDSC/JobPlan payload in the `.pte`.

In [ ]:
from executorch.exir import to_edge

ARTIFACTS = Path(os.environ.get("EXECUTORCH_SPYRE_ARTIFACTS", work_home / "artifacts"))
ARTIFACTS.mkdir(parents=True, exist_ok=True)

exported = torch.export.export(model, (a, b))
edge = to_edge(exported)
delegated = edge.to_backend(SpyrePartitioner())
program = delegated.to_executorch()

pte_path = ARTIFACTS / "spyre_matmul.pte"
pte_path.write_bytes(program.buffer)
a.contiguous().cpu().numpy().tofile(ARTIFACTS / "a.bin")
b.contiguous().cpu().numpy().tofile(ARTIFACTS / "b.bin")
reference.contiguous().cpu().numpy().tofile(ARTIFACTS / "ref.bin")
print(f"Wrote {pte_path} ({pte_path.stat().st_size:,} bytes)")
print(f"Artifacts saved to: {ARTIFACTS}")

## 5. Inspect the delegated artifact

Confirms that preprocessing produced a non-empty `.pte` with an ExecuTorch
delegate call. Device execution is verified in Part 2.

In [ ]:
graph = delegated.exported_program().graph
print(graph)
delegate_nodes = [
    node for node in graph.nodes
    if "delegate" in str(node.target).lower() or "delegate" in node.name.lower()
]
assert delegate_nodes, "No delegate call found — graph was not delegated to Spyre"

for name in ("spyre_matmul.pte", "a.bin", "b.bin", "ref.bin"):
    path = ARTIFACTS / name
    assert path.is_file() and path.stat().st_size > 0, f"Missing or empty: {path}"
    print(f"{name:20s} {path.stat().st_size:>10,} bytes")
print("Delegate nodes:", len(delegate_nodes))
print()
print("✅ .pte generated successfully. Open 02_run_on_device.ipynb to execute it on physical Spyre.")

## 6. Release resources and tear down

`free_runtime()` tears down the Python-side runtime object.

> **Important:** Shut down this notebook's kernel (**Kernel → Shut Down Kernel** in JupyterLab)
> before opening Part 2, as VFIO device file descriptors are released when the process terminates.

In [ ]:
import torch_spyre
if hasattr(torch_spyre, "_C") and hasattr(torch_spyre._C, "free_runtime"):
    torch_spyre._C.free_runtime()
    print("Spyre runtime released.")
else:
    print("Runtime already clear.")

print("➡ Next step: Shut down this kernel (Kernel → Shut Down Kernel), then open 02_run_on_device.ipynb.")